Results Figures:

In [ ]:
import json
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path
from sklearn.metrics import roc_curve, auc
from scipy.interpolate import pchip_interpolate

# ==========================================
# 1. GLOBAL PUBLICATION STYLE (LaTeX/Serif)
# ==========================================
# Consistent colors: Green for Diverticulitis (#2E7D32), Red for Cancer (#C62828)
COLOR_DIVERT = "#2E7D32" 
COLOR_CANCER = "#C62828"
COLOR_GLOBAL = "#1C4E80" # Deep Blue for Global/ROC

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 14,
    "axes.labelsize": 16,
    "axes.titlesize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 1.2,
    "legend.fontsize": 13,
    "mathtext.fontset": "stix",
})

# ==========================================
# 2. METRICS PLOTTING (nnU-Net Results)
# ==========================================
def plot_segmentation_metrics(json_path, class_csv):
    with open(json_path, "r") as f:
        results = json.load(f)
    
    per_case = results["per_case"]
    rows = [{"uid": str(uid), **v["before"]} for uid, v in per_case.items()]
    df = pd.DataFrame(rows)
    
    class_df = pd.read_csv(class_csv)
    class_df["UID"] = class_df["UID"].astype(str)
    df = df.merge(class_df, left_on="uid", right_on="UID", how="left")

    metrics = [("dice", "Dice Coefficient"), ("precision", "Precision"), ("recall", "Recall")]
    
    for metric_key, ylabel in metrics:
        class_data = [df[df["target"] == 0][metric_key], df[df["target"] == 1][metric_key]]
        
        fig, ax = plt.subplots(figsize=(4.5, 6), dpi=100)
        
        bp = ax.boxplot(
            class_data,
            labels=["Diverticulitis", "Colon Cancer"],
            patch_artist=True,
            widths=0.6,
            medianprops=dict(color="black", linewidth=2),
            boxprops=dict(linewidth=1.5)
        )

        # Apply standardized colors
        bp["boxes"][0].set_facecolor(COLOR_DIVERT)
        bp["boxes"][1].set_facecolor(COLOR_CANCER)
        bp["boxes"][0].set_alpha(0.7)
        bp["boxes"][1].set_alpha(0.7)

        # Style tick labels
        for tick, color in zip(ax.get_xticklabels(), [COLOR_DIVERT, COLOR_CANCER]):
            tick.set_color(color)
            tick.set_fontweight("bold")

        ax.set_ylabel(ylabel, fontweight="bold")
        #ax.set_title(f"Segmentation Performance: {ylabel}", pad=15)
        global_mean = df[metric_key].median()
        ax.text(
            0.15, 1.05, 
            f"Global Median: {global_mean:.3f}", 
            transform=ax.transAxes, 
            verticalalignment='top',
            fontweight='bold',
            fontsize=14,
            color=COLOR_GLOBAL,
            bbox=dict(
                facecolor='white', 
                edgecolor=COLOR_GLOBAL, 
                boxstyle='round,pad=0.5',
                alpha=0.9,
                linewidth=1.5
            )
        )
        ax.grid(axis='y', linestyle='--', alpha=0.3)
        
        plt.tight_layout()
        plt.show()

# ==========================================
# 3. SMOOTH ROC CURVE (Classification Results)
# ==========================================
def plot_smooth_roc(csv_path):
    df = pd.read_csv(csv_path)
    y_true = df["GT"].values
    y_scores = df["NN_pred"].apply(lambda x: ast.literal_eval(x)[1]).values

    fpr, tpr, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)

    # Monotonic smoothing
    _, idx = np.unique(fpr, return_index=True)
    fpr_smooth = np.linspace(0, 1, 200)
    tpr_smooth = np.clip(pchip_interpolate(fpr[idx], tpr[idx], fpr_smooth), 0, 1)

    fig, ax = plt.subplots(figsize=(6, 6), dpi=120)
    ax.fill_between(fpr_smooth, tpr_smooth, alpha=0.1, color=COLOR_GLOBAL)
    ax.plot(
        fpr_smooth, 
        tpr_smooth, 
        color=COLOR_GLOBAL, 
        lw=2.5, 
        label=rf"$\mathbf{{Model\ (AUC = {roc_auc:.3f})}}$",
        zorder=3
    )

    # Reference "Random Chance" Line (Now added to legend)
    ax.plot(
        [0, 1], [0, 1], 
        linestyle="--", 
        color="#888888", 
        lw=1.5, 
        alpha=0.8, 
        label="Random Classifier (AUC = 0.50)",
        zorder=2
    )

    ax.set_xlim(-0.01, 1.0)
    ax.set_ylim(0.0, 1.02)
    ax.set_xlabel("False Positive Rate (1 − Specificity)", labelpad=10, fontdict={"fontweight": "bold"})
    ax.set_ylabel("True Positive Rate (Sensitivity)", labelpad=10, fontdict={"fontweight": "bold"})
    ax.legend(loc="lower right", frameon=False)
    #ax.set_title("Classification ROC Curve", pad=15)
    
    plt.tight_layout()
    plt.savefig("ROC.pdf", format="pdf", bbox_inches="tight")
    plt.show()

# ==========================================
# 4. CONFUSION MATRIX (Aligned Labels)
# ==========================================
def plot_consistent_cm():
    #cm = np.array([[26, 6], [1, 44]])
    cm = np.array([[13, 19], [0, 45]])
    class_names = ["Diverticulitis", "Colon Cancer"]
    
    fig, ax = plt.subplots(figsize=(6, 6), dpi=120)
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues, alpha=0.8)

    #ax.set_title("Confusion Matrix: Differential Diagnosis", pad=25, fontweight='bold')
    ax.set_ylabel("True Pathology", labelpad=20, fontweight='bold')
    ax.set_xlabel("Predicted Pathology", labelpad=20, fontweight='bold')

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(class_names)
    ax.set_yticklabels(class_names, rotation=90, va='center')

    # Colorize labels
    for i, tick in enumerate(ax.get_xticklabels()):
        tick.set_color([COLOR_DIVERT, COLOR_CANCER][i])
        tick.set_fontweight('bold')
    for i, tick in enumerate(ax.get_yticklabels()):
        tick.set_color([COLOR_DIVERT, COLOR_CANCER][i])
        tick.set_fontweight('bold')

    # Cell values
    thresh = cm.max() / 1.5
    for i in range(2):
        for j in range(2):
            ax.text(j, i, format(cm[i, j], 'd'), ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black",
                    fontsize=18, fontweight='bold')

    #fig.colorbar(im, ax=ax, fraction=0.046, pad=0.1).outline.set_visible(False)
    plt.tight_layout()
    plt.savefig("CM.pdf", format="pdf", bbox_inches="tight")
    plt.show()

#csv_path = ("/data/colon_cancer/CC_Detection/Resnet_results/inf_outputs/ResNet_Dataset110_CC_2026_02_11_172950/Dataset110_CC/results.csv")
csv_path = "/data/colon_cancer/CC_Detection/Resnet_results/inf_outputs/ResNet_Dataset109_CC_2026_02_08_215318/Dataset111_CC/results.csv"



json_path = "/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/final/results_Ts_pp.json"
class_csv = "/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/splits.csv"
# ==========================================
# EXECUTION
# ==========================================
#plot_segmentation_metrics(json_path, class_csv) # Uncomment to run
plot_smooth_roc(csv_path)                      # Uncomment to run
plot_consistent_cm()

Qualitative Results:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
import matplotlib as mpl
import pandas as pd
from pathlib import Path

# ============================================================
# Global Style Consistency
# ============================================================
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 14,
    "mathtext.fontset": "stix",
    
    # --- ADD THESE TWO LINES ---
    "pdf.fonttype": 42,  # Output Type 42 (TrueType) fonts for PDF
    "ps.fonttype": 42    # Output Type 42 fonts for PostScript
    # ---------------------------
})

# Neutral Mask Colors (to distinguish GT from Pred)
COLOR_GT = [1.0, 0.57, 0.0]   # Gold
COLOR_PRED = [0.0, 0.69, 1.0]  # Blue

# Pathology Label Colors (Consistent with Stats/CM)
PATH_COLORS = {0: "#2E7D32", 1: "#C62828"} # 0: Div (Green), 1: Cancer (Red)
PATH_NAMES = {0: "Diverticulitis", 1: "Colon Cancer"}

# ============================================================
# Helper Functions (Windowing & BBox)
# ============================================================

def load_volume(path):
    path = Path(path)
    if path.suffix == ".npy": return np.load(path)
    return nib.load(str(path)).get_fdata()

def load_case(uid, img_dir, gt_dir, pred_dir):
    img = load_volume(Path(img_dir) / f"{uid}_0000.nii.gz")
    gt = load_volume(Path(gt_dir) / f"{uid}.nii.gz")
    pred = load_volume(Path(pred_dir) / f"{uid}.nii.gz")
    return img, gt, pred

def window_ct_hu(ct, level=50, width=350):
    lo, hi = level - width / 2, level + width / 2
    return (np.clip(ct, lo, hi) - lo) / (hi - lo + 1e-6)

def compute_square_bbox(mask, axis, margin=60):
    coords = np.where(mask > 0)
    axes_2d = [i for i in range(3) if i != axis]
    y_min, y_max = coords[axes_2d[0]].min(), coords[axes_2d[0]].max()
    x_min, x_max = coords[axes_2d[1]].min(), coords[axes_2d[1]].max()
    size = max(y_max - y_min, x_max - x_min) + margin
    cy, cx = (y_min + y_max) // 2, (x_min + x_max) // 2
    return max(0, cy - size // 2), cy + size // 2, max(0, cx - size // 2), cx + size // 2

def crop_and_rotate(slice2d, bbox):
    y0, y1, x0, x1 = bbox
    return np.rot90(slice2d[y0:y1, x0:x1], k=1)

def overlay_single(ax, ct, mask, color, alpha=0.5):
    ax.imshow(ct, cmap="gray")
    overlay = np.zeros((*mask.shape, 4))
    overlay[..., :3] = color
    overlay[..., 3] = mask * alpha
    ax.imshow(overlay)
    ax.axis("off")

# ============================================================
# Plot & Save
# ============================================================

def plot_and_save_sample(ct, gt, pred, uid, slice_ids, case_idx, target_class, axis=2, save_dir="figures"):
    save_dir = Path(save_dir); save_dir.mkdir(parents=True, exist_ok=True)
    gt_bin, pred_bin = gt > 0, pred > 0
    bbox = compute_square_bbox(gt_bin | pred_bin, axis)

    fig, axes = plt.subplots(len(slice_ids), 2, figsize=(6, 3 * len(slice_ids)), dpi=150)
    if len(slice_ids) == 1: axes = axes[None, :]

    # Top Column Titles
    axes[0, 0].set_title("Ground Truth", fontsize=14, fontweight="bold", pad=10, color=COLOR_GT)
    axes[0, 1].set_title("Model Prediction", fontsize=14, fontweight="bold", pad=10, color=COLOR_PRED)

    for i, z in enumerate(slice_ids):
        sl_idx = [slice(None)] * 3
        sl_idx[axis] = z
        
        ct_sl = crop_and_rotate(window_ct_hu(ct[tuple(sl_idx)]), bbox)
        gt_sl = crop_and_rotate(gt_bin[tuple(sl_idx)], bbox)
        pr_sl = crop_and_rotate(pred_bin[tuple(sl_idx)], bbox)

        overlay_single(axes[i, 0], ct_sl, gt_sl, COLOR_GT)
        overlay_single(axes[i, 1], ct_sl, pr_sl, COLOR_PRED)

        axes[i, 0].text(-0.15, 0.5, f"Slice {z}", transform=axes[i, 0].transAxes, 
                        rotation=90, va="center", ha="right", fontsize=14, fontweight="bold")

    # Bottom Title: Case Number and Pathology
    path_name = PATH_NAMES.get(target_class, "Unknown")
    path_color = PATH_COLORS.get(target_class, "black")
    
    # Place text at the very bottom
    fig.text(0.5, 0.02, f"Case 8: {path_name}", 
             ha="center", va="bottom", fontsize=14, fontweight="bold", 
             color=path_color, bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=3))

    plt.subplots_adjust(wspace=0.05, hspace=0.05, bottom=0.08)
    
    out_path = save_dir / f"qualitative_case{case_idx}_{uid}.pdf"
    plt.savefig(out_path, bbox_inches="tight", transparent=True)
    plt.show()

# ============================================================
# Interface
# ============================================================

def visualize_uids(uids, img_dir, gt_dir, pred_dir, label_csv, axis=2, slice_map=None, save_dir="figures"):
    labels_df = pd.read_csv(label_csv)
    labels_df['UID'] = labels_df['UID'].astype(str)

    for idx, uid in enumerate(uids, start=1): # case_idx starts at 1
        print(f"Processing Case {idx} (UID {uid})...")
        ct, gt, pred = load_case(uid, img_dir, gt_dir, pred_dir)
        
        # Get pathology class for coloring
        row = labels_df[labels_df['UID'] == str(uid)]
        target_class = int(row['target'].values[0]) if not row.empty else 0
        
        slices = slice_map[uid] if slice_map and uid in slice_map else [gt.shape[axis]//2]
        plot_and_save_sample(ct, gt, pred, uid, slices, idx, target_class, axis=axis, save_dir=save_dir)

# ============================================================
# Run
# ============================================================

visualize_uids(
    #uids = [28, 344, 13, 239],
    #slice_map = {28: [132, 154, 172], 344: [31, 35, 41], 13: [23, 42, 46], 239: [23, 25, 29]},
    uids=[251],
    #uids = [351,28,172,102,13,239,528,628],
    slice_map = {351:[28,37,40], 28: [ 132,154, 172], 13:[23,42,46], 239:[23,25,29], 172:[54,69,76],102:[146,154,162], 628:[28,30,33], 528:[25,28,30], 344:[31,35,41],251:[25,27,29]},
    img_dir = "/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/imagesTs",
    gt_dir = "/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/labelsTs",
    pred_dir = "/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/predictionsTs",
    label_csv = "/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/labels.csv", 
    save_dir = "figures"
)

segmentation-based performance


In [ ]:
import json

# Load JSON file

with open("/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/final/results_Ts_pp.json", "r") as f:
    data = json.load(f)

per_case = data["per_case"]
mean_dice = data["summary"]["mean_before"]["dice"]

good_ids = []
bad_ids = []

for case_id, metrics in per_case.items():
    dice = metrics["before"]["dice"]
    if dice >= mean_dice:
        good_ids.append(case_id)
    else:
        print(dice)
        bad_ids.append(case_id)

print("Mean Dice (before):", mean_dice)
print("Good cases:", good_ids)
print("Bad cases:", bad_ids)

In [ ]:
import json
import pandas as pd
from sklearn.metrics import confusion_matrix, accuracy_score

# =========================
# Paths
# =========================
SEG_JSON = "/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/final/results_Ts_pp.json"
CLS_CSV  = "/data/colon_cancer/CC_Detection/Resnet_results/inf_outputs/ResNet_Dataset110_CC_2026_02_11_172950/Dataset110_CC/results.csv"

# =========================
# Load segmentation results
# =========================
with open(SEG_JSON, "r") as f:
    seg_data = json.load(f)

per_case = seg_data["per_case"]
mean_dice = seg_data["summary"]["mean_before"]["dice"]

good_ids = []
bad_ids = []

for uid, metrics in per_case.items():
    dice = metrics["before"]["dice"]
    if dice >= mean_dice:
        good_ids.append(uid)
    else:
        bad_ids.append(uid)

good_ids = set(good_ids)
bad_ids = set(bad_ids)

print(f"Mean Dice (before): {mean_dice:.4f}")
print(f"# Good segmentations: {len(good_ids)}")
print(f"# Bad segmentations:  {len(bad_ids)}")

# =========================
# Load classification results
# =========================
df = pd.read_csv(CLS_CSV)
df["UID"] = df["UID"].astype(str)

# =========================
# Metric function
# =========================
def compute_metrics(df_subset, name):
    y_true = df_subset["GT"]
    y_pred = df_subset["NN"]

    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    print(f"\n{name}")
    print(f"Accuracy: {acc:.4f}")
    print("Confusion matrix [[TN FP], [FN TP]]:")
    print(cm)

    return acc, cm

# =========================
# All cases
# =========================
compute_metrics(df, "ALL CASES")

# =========================
# Good segmentations
# =========================
df_good = df[df["UID"].isin(good_ids)]
compute_metrics(df_good, "GOOD SEGMENTATIONS")

# =========================
# Bad segmentations
# =========================
df_bad = df[df["UID"].isin(bad_ids)]
compute_metrics(df_bad, "BAD SEGMENTATIONS")

# =========================
# Optional: sanity check
# =========================
missing = set(per_case.keys()) - set(df["UID"])
if missing:
    print("\nWARNING: Missing classification results for these UIDs:")
    print(sorted(missing))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Color Definitions
COLOR_DIVERT = "#2ca02c" # Green
COLOR_CANCER = "#d62728" # Red
COLOR_SEG_TITLE = "#000080" # Grey for "Good/Poor"
COLOR_AXIS_LABEL = "#000000" # Black for "True/Predicted"

def plot_good_bad_confusion_matrices(cm_good, cm_bad):
    class_names = ["Div.", "CC"]
    cms = [cm_good, cm_bad]
    titles = ["Good Segmentation", "Poor Segmentation"]
    vmax = max(cm_good.max(), cm_bad.max())

    # figsize tuned for a 1-column paper width
    fig, axes = plt.subplots(
        2, 1, 
        figsize=(3.2, 5.0),  
        gridspec_kw={"hspace": 0.25}
    )

    for i, (ax, cm, title) in enumerate(zip(axes, cms, titles)):
        im = ax.imshow(cm, cmap=plt.cm.Blues, vmin=0, vmax=vmax, alpha=0.8)
        
        # 1. Individual Row Titles (Good/Poor) - Centered on each matrix
        ax.set_ylabel(title, fontsize=9, fontweight="bold", 
                      color=COLOR_SEG_TITLE, labelpad=12)
        
        # 2. Tick & Label Formatting
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        
        # Y-Axis Class Labels (Directly next to the boxes)
        ax.set_yticklabels(class_names, fontweight="bold", fontsize=9, rotation=90, va='center')
        for tick, color in zip(ax.get_yticklabels(), [COLOR_DIVERT, COLOR_CANCER]):
            tick.set_color(color)

        # X-Axis Class Labels (Bottom only)
        
        ax.set_xticklabels(class_names, fontweight="bold", fontsize=9)
        for tick, color in zip(ax.get_xticklabels(), [COLOR_DIVERT, COLOR_CANCER]):
            tick.set_color(color)
        

        # 3. Cell Text
        thresh = vmax * 0.5
        for row in range(2):
            for col in range(2):
                ax.text(col, row, cm[row, col],
                        ha="center", va="center",
                        fontsize=11, fontweight="bold",
                        color="white" if cm[row, col] > thresh else "black")
        
        for spine in ax.spines.values():
            spine.set_visible(False)

    # 4. Global Positioning
    # True Label: Positioned at the far center-left
    fig.text(0.18, 0.5, "True Label", va="center", rotation="vertical", 
             fontweight="bold", fontsize=11, color=COLOR_AXIS_LABEL)
    
    # Predicted Label: Positioned at the bottom center
    fig.text(0.53, 0.02, "Predicted Label", ha="center", 
             fontweight="bold", fontsize=11, color=COLOR_AXIS_LABEL)

    # Compact Colorbar
    cbar = fig.colorbar(im, ax=axes, fraction=0.046, pad=0.06)
    cbar.ax.tick_params(labelsize=8)
    cbar.outline.set_visible(False)

    # Adjust layout to make room for the far-left label
    plt.tight_layout(rect=[0.08, 0.03, 1, 1])
    plt.show()

# Data
cm_good = np.array([[19, 1], [0, 33]])
cm_bad = np.array([[7, 5], [1, 11]])

plot_good_bad_confusion_matrices(cm_good, cm_bad)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Color Definitions
COLOR_DIVERT = "#2ca02c"  # Green
COLOR_CANCER = "#d62728"  # Red
COLOR_SEG_TITLE = "#000080" # Navy Blue
COLOR_AXIS_LABEL = "#000000" # Black

def plot_triple_confusion_matrix(cm_global, cm_good, cm_bad):
    class_names = ["Div.", "CC"]
    vmax = max(cm_global.max(), cm_good.max(), cm_bad.max())
    
    # Create a 2x2 grid. 
    fig = plt.figure(figsize=(7, 5))
    ax_global = plt.subplot2grid((2, 2), (0, 0), rowspan=2)
    ax_good = plt.subplot2grid((2, 2), (0, 1))
    ax_bad = plt.subplot2grid((2, 2), (1, 1))

    axes_list = [ax_global, ax_good, ax_bad]
    cms = [cm_global, cm_good, cm_bad]
    titles = ["Global Confusion Matrix", "Good Segmentation", "Poor Segmentation"]

    for i, (ax, cm, title) in enumerate(zip(axes_list, cms, titles)):
        im = ax.imshow(cm, cmap=plt.cm.Blues, vmin=0, vmax=vmax, alpha=0.8)
        
        # 1. Row Titles (Navy Blue)
        ax.set_ylabel(title, fontsize=9, fontweight="bold", 
                      color=COLOR_SEG_TITLE, labelpad=25)
        
        # 2. Tick & Label Formatting
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        
        # Y-Axis (True) Class Labels
        ax.set_yticklabels(class_names, fontweight="bold", fontsize=9, rotation=90, va='center')
        for tick, color in zip(ax.get_yticklabels(), [COLOR_DIVERT, COLOR_CANCER]):
            tick.set_color(color)

        # X-Axis (Predicted) Class Labels
        ax.set_xticklabels(class_names, fontweight="bold", fontsize=9)
        for tick, color in zip(ax.get_xticklabels(), [COLOR_DIVERT, COLOR_CANCER]):
            tick.set_color(color)

        # 3. Cell Text
        thresh = vmax * 0.5
        for row in range(2):
            for col in range(2):
                ax.text(col, row, cm[row, col],
                        ha="center", va="center",
                        fontsize=12 if i == 0 else 10,
                        fontweight="bold",
                        color="white" if cm[row, col] > thresh else "black")
        
        # 4. Add Frames (Spines)
        # We turn them back on and set a subtle color/linewidth
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color('#333333')
            spine.set_linewidth(1.0)

    # 5. Structural Labels
    # True Label (Far Left)
    fig.text(0.02, 0.5, "True Label", va="center", rotation="vertical", 
             fontweight="bold", fontsize=11, color=COLOR_AXIS_LABEL)
    
    # Predicted Labels (Centered under the two vertical columns)
    # 0.28 is roughly the center of the left column, 0.72 for the right
    fig.text(0.5, 0.04, "Predicted Label", ha="center", fontweight="bold", fontsize=10)
    #fig.text(0.72, 0.04, "Predicted Label", ha="center", fontweight="bold", fontsize=10)

    plt.tight_layout(rect=[0.05, 0.05, 1, 0.95])
    plt.show()

# Data
cm_global = np.array([[26, 6], [1, 44]])
cm_good = np.array([[19, 1], [0, 33]])
cm_bad = np.array([[7, 5], [1, 11]])

plot_triple_confusion_matrix(cm_global, cm_good, cm_bad)

In [ ]:
import pickle

def inspect_pkl(path):
    with open(path, "rb") as f:
        obj = pickle.load(f)

    print(f"\n📦 PKL file: {path}")
    print("Top-level type:", type(obj))

    if isinstance(obj, dict):
        print("Keys:")
        for k, v in obj.items():
            print(f"  - {k}: type={type(v)}")
    elif isinstance(obj, (list, tuple)):
        print(f"Length: {len(obj)}")
        print("First element type:", type(obj[0]) if len(obj) > 0 else None)

    return obj

import numpy as np

def inspect_npz(path):
    data = np.load(path, allow_pickle=True)

    print(f"\n📦 NPZ file: {path}")
    print("Keys:")
    for k in data.files:
        v = data[k]
        print(f"  - {k}: type={type(v)}, shape={getattr(v, 'shape', None)}, dtype={getattr(v, 'dtype', None)}")
    print(data["probabilities"][1:].shape)

    return data


inspect_pkl("/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/final/predictionsTr/1.pkl")
inspect_npz("/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/final/predictionsTr/1.npz")

In [ ]:
import numpy as np
from pathlib import Path

# ==========================================
# Utility: Softmax
# ==========================================
def softmax(x, axis=0):
    x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

# ==========================================
# Checks
# ==========================================
def check_value_range(arr):
    min_val = arr.min()
    max_val = arr.max()
    print(f"Value range: min={min_val:.6f}, max={max_val:.6f}")

    if min_val >= 0 and max_val <= 1:
        print("✓ Values in [0, 1] → likely probabilities")
    else:
        print("✗ Values outside [0, 1] → likely logits")

def check_sum_to_one(arr, class_axis=0, atol=1e-3):
    sums = np.sum(arr, axis=class_axis)
    print(f"Sum over class axis={class_axis}:")
    print(f"  min={sums.min():.6f}, max={sums.max():.6f}, mean={sums.mean():.6f}")

    if np.allclose(sums, 1.0, atol=atol):
        print("✓ Sums ≈ 1 → softmax probabilities")
        return True
    else:
        print("✗ Sums ≠ 1 → logits or unnormalized scores")
        return False

def check_negative_values(arr):
    has_neg = np.any(arr < 0)
    print(f"Contains negative values: {has_neg}")
    if has_neg:
        print("✗ Negative values → logits")
    else:
        print("✓ No negative values → could be probabilities")

def argmax_invariance(arr, class_axis=0):
    arg1 = np.argmax(arr, axis=class_axis)
    arg2 = np.argmax(softmax(arr, axis=class_axis), axis=class_axis)

    agreement = np.mean(arg1 == arg2)
    print(f"Argmax agreement before/after softmax: {agreement * 100:.2f}%")

    if agreement < 1.0:
        print("✗ Argmax changed → input was logits")
    else:
        print("✓ Argmax unchanged → consistent with probabilities")

# ==========================================
# Main inspection function
# ==========================================
def inspect_npz_softmax_vs_logits(npz_path, class_axis=0):
    npz_path = Path(npz_path)
    assert npz_path.exists(), f"File not found: {npz_path}"

    data = np.load(npz_path, allow_pickle=True)

    print("=" * 80)
    print(f"Inspecting NPZ file: {npz_path}")
    print("Available keys:", data.files)
    print("=" * 80)

    for key in data.files:
        arr = data[key]

        if not isinstance(arr, np.ndarray):
            print(f"\nSkipping key '{key}' (not a numpy array)")
            continue

        print(f"\nKey: '{key}'")
        print(f"Shape: {arr.shape}, dtype: {arr.dtype}")

        # Heuristic: class axis must exist
        if arr.ndim < 2:
            print("Skipping (not a multi-class tensor)")
            continue

        check_value_range(arr)
        check_negative_values(arr)
        is_softmax = check_sum_to_one(arr, class_axis=class_axis)
        argmax_invariance(arr, class_axis=class_axis)

        print("→ Final verdict:",
              "SOFTMAX PROBABILITIES" if is_softmax else "LIKELY LOGITS")

        print("-" * 80)

# ==========================================
# Example usage
# ==========================================
if __name__ == "__main__":
    npz_file = "/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/final/predictionsTr/1.npz"   # <-- CHANGE THIS
    class_axis = 0                # nnU-Net default: (C, H, W, D)

    inspect_npz_softmax_vs_logits(npz_file, class_axis)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.utils import resample

def calculate_all_metrics(y_true, y_pred):
    """Calculates Accuracy, Sensitivity, and Specificity."""
    acc = accuracy_score(y_true, y_pred)
    # confusion_matrix returns [[TN, FP], [FN, TP]] for binary classification
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    return acc, sensitivity, specificity

def run_full_bootstrap(my_results_path, baseline_results_path, n_iterations=1000):
    # 1. Load and Merge Data
    df_mine = pd.read_csv(my_results_path)
    df_base = pd.read_csv(baseline_results_path)
    
    # Ensure UIDs match so we compare the same patients in each bootstrap row
    df = pd.merge(df_mine, df_base[['UID', 'NN']], on='UID', suffixes=('_mine', '_base'))
    
    boot_data = []
    print(f"Bootstrapping {n_iterations} iterations for 77 samples...")

    for i in range(n_iterations):
        # Resample with replacement
        sample = resample(df, replace=True, n_samples=len(df))
        
        # Calculate metrics for My Model
        acc_m, sens_m, spec_m = calculate_all_metrics(sample['GT'], sample['NN_mine'])
        # Calculate metrics for Baseline
        acc_b, sens_b, spec_b = calculate_all_metrics(sample['GT'], sample['NN_base'])
        
        boot_data.append({
            'acc_m': acc_m, 'sens_m': sens_m, 'spec_m': spec_m,
            'acc_b': acc_b, 'sens_b': sens_b, 'spec_b': spec_b,
            'diff_acc': acc_m - acc_b,
            'diff_sens': sens_m - sens_b,
            'diff_spec': spec_m - spec_b
        })

    boot_df = pd.DataFrame(boot_data)

    # 2. Reporting Function
    def print_metric_results(name, my_col, base_col, diff_col):
        mean_m = boot_df[my_col].mean()
        ci_m = (boot_df[my_col].quantile(0.025), boot_df[my_col].quantile(0.975))
        
        mean_b = boot_df[base_col].mean()
        ci_b = (boot_df[base_col].quantile(0.025), boot_df[base_col].quantile(0.975))
        
        # P-value: how often was the baseline better or equal to my model?
        p_val = (boot_df[diff_col] <= 0).mean()
        
        
        print(f"\n--- {name} ---")
        print(f"Proposed: {mean_m:.3f} ({ci_m[0]:.3f} - {ci_m[1]:.3f})")
        print(f"Baseline: {mean_b:.3f} ({ci_b[0]:.3f} - {ci_b[1]:.3f})")
        print(f"P-value:  {p_val:.4f} {'*' if p_val < 0.05 else ''}")

    print("\n" + "="*50)
    print("STATISTICAL SIGNIFICANCE REPORT (Bootstrapped)")
    print("="*50)
    
    print_metric_results("ACCURACY", "acc_m", "acc_b", "diff_acc")
    print_metric_results("SENSITIVITY", "sens_m", "sens_b", "diff_sens")
    print_metric_results("SPECIFICITY", "spec_m", "spec_b", "diff_spec")
    
    print("\n" + "="*50)

if __name__ == "__main__":
    # Update these paths to your actual CSV files
    run_full_bootstrap("/data/colon_cancer/CC_Detection/Resnet_results/inf_outputs/ResNet_Dataset110_CC_2026_02_11_172950/Dataset110_CC/results.csv", "/data/colon_cancer/CC_Detection/Resnet_results/inf_outputs/ResNet_Dataset110_CC_2026_02_11_183514/Dataset110_CC/results.csv")

In [ ]:
run_full_bootstrap("/data/colon_cancer/CC_Detection/Resnet_results/inf_outputs/ResNet_Dataset110_CC_2026_02_11_172950/Decathlon/results.csv", "/data/colon_cancer/CC_Detection/Resnet_results/inf_outputs/ResNet_Dataset110_CC_2026_02_11_183514/Decathlon/results.csv")

In [ ]:
run_full_bootstrap("/data/colon_cancer/CC_Detection/Resnet_results/inf_outputs/ResNet_Dataset110_CC_2026_02_11_172950/stage_2_cc/results.csv", "/data/colon_cancer/CC_Detection/Resnet_results/inf_outputs/ResNet_Dataset110_CC_2026_02_11_183514/stage_2_cc/results.csv")

In [ ]:
import pandas as pd
import numpy as np
import ast  # safe string-to-tuple conversion

def load_and_extract_spacings(csv_path):
    df = pd.read_csv(csv_path)
    
    # Convert string "(x, y, z)" → tuple (x, y, z)
    spacings = df['original_shape'].apply(ast.literal_eval)
    
    # Convert to Nx3 numpy array
    return np.vstack(spacings.values)

# Paths to your two files
csv1 = "/data/colon_cancer/CC_Detection/pp_data/Dataset110_CC/cropping_metadata_Tr.csv"
csv2 = "/data/colon_cancer/CC_Detection/pp_data/Dataset110_CC/cropping_metadata_Ts.csv"

# Load spacings
spacings1 = load_and_extract_spacings(csv1)
spacings2 = load_and_extract_spacings(csv2)

# Combine both datasets
all_spacings = np.vstack([spacings1, spacings2])

# Compute global mean spacing
mean_spacing = all_spacings.mean(axis=0)

print(f"Global mean original spacing: {mean_spacing}")